# Forecasting Model: Testing Same-Month-Last-Year (Lag12) Feature

This notebook extends the forecasting model from notebook 15 by testing whether adding `delay_rate_lag12` (same month last year) improves predictions. The hypothesis is that monthly delay patterns may have yearly seasonality beyond what `month_sin/cos` encoding captures. For example:
- December 2024 delays might correlate with December 2023 delays
- Route-specific seasonal patterns (e.g., Hobart in winter) might be captured better by explicit lag12

The trade-off is losing 12 months of data per airline-route from the training data (first year has no lag12).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed")

%matplotlib inline

## 1. Data Preparation

Same filtering as notebook 15.

In [ ]:
# Load data
df = pd.read_csv('../data/processed/ml_training_data_multiroute_hols.csv')

df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print("Original shape: {}".format(df.shape))
print("Date range: {} to {}".format(df['year_month'].min(), df['year_month'].max()))

In [ ]:
# Filter low-volume and anomalous routes (same as notebook 15)
volume_threshold = 40
airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean()
high_volume_ar = airline_route_volume[airline_route_volume >= volume_threshold].index.tolist()
df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()

anomalous_routes = ['Melbourne_Hobart', 'Adelaide_Sydney', 'Perth_Brisbane']
df_filtered = df_filtered[~df_filtered['route'].isin(anomalous_routes)].copy()

print("Records after filtering: {}".format(len(df_filtered)))

## 2. Feature Engineering

In [ ]:
# Standard lag features
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']

# NEW: Same-month-last-year (lag12)
df_filtered['delay_rate_lag12'] = df_filtered.groupby('airline_route')['delay_rate'].shift(12)

# Cyclical month encoding
df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

# One-hot encoding
airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = list(route_dummies.columns)

print("Lag12 created. Checking correlation with target...")

In [ ]:
# Check correlation of lag12 with target
lag_corrs = df_filtered[['delay_rate', 'delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_lag12', 'delay_rate_gradient']].corr()

print("Correlation with delay_rate:")
print("-" * 40)
print("  delay_rate_lag1:     {:.4f}".format(lag_corrs.loc['delay_rate', 'delay_rate_lag1']))
print("  delay_rate_lag2:     {:.4f}".format(lag_corrs.loc['delay_rate', 'delay_rate_lag2']))
print("  delay_rate_lag12:    {:.4f}".format(lag_corrs.loc['delay_rate', 'delay_rate_lag12']))
print("  delay_rate_gradient: {:.4f}".format(lag_corrs.loc['delay_rate', 'delay_rate_gradient']))

In [ ]:
# Visualize lag12 correlation
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter: lag1 vs target
ax = axes[0]
valid_lag1 = df_filtered.dropna(subset=['delay_rate', 'delay_rate_lag1'])
ax.scatter(valid_lag1['delay_rate_lag1'], valid_lag1['delay_rate'], alpha=0.3, s=10)
ax.plot([0, 0.6], [0, 0.6], 'r--', label='Perfect correlation')
ax.set_xlabel('delay_rate_lag1 (previous month)')
ax.set_ylabel('delay_rate (current month)')
ax.set_title('Lag1 vs Target (r = {:.3f})'.format(lag_corrs.loc['delay_rate', 'delay_rate_lag1']))
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.6)
ax.set_ylim(0, 0.6)

# Scatter: lag12 vs target
ax = axes[1]
valid_lag12 = df_filtered.dropna(subset=['delay_rate', 'delay_rate_lag12'])
ax.scatter(valid_lag12['delay_rate_lag12'], valid_lag12['delay_rate'], alpha=0.3, s=10)
ax.plot([0, 0.6], [0, 0.6], 'r--', label='Perfect correlation')
ax.set_xlabel('delay_rate_lag12 (same month last year)')
ax.set_ylabel('delay_rate (current month)')
ax.set_title('Lag12 vs Target (r = {:.3f})'.format(lag_corrs.loc['delay_rate', 'delay_rate_lag12']))
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.6)
ax.set_ylim(0, 0.6)

plt.tight_layout()
plt.show()

## 3. Compare Data Loss

Adding lag12 means losing the first 12 months of data per airline-route.

In [ ]:
# Compare data availability
df_baseline = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).copy()
df_with_lag12 = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient', 'delay_rate_lag12']).copy()

print("Data availability comparison:")
print("-" * 50)
print("  Baseline (lag1, lag2, gradient): {:,} rows".format(len(df_baseline)))
print("  With lag12:                      {:,} rows".format(len(df_with_lag12)))
print("  Data loss:                       {:,} rows ({:.1f}%)".format(
    len(df_baseline) - len(df_with_lag12),
    (len(df_baseline) - len(df_with_lag12)) / len(df_baseline) * 100
))

## 4. Train/Test Split

In [ ]:
# Use df_with_lag12 for fair comparison (same data for both models)
df_clean = df_with_lag12.copy()

train_mask = (((df_clean['year'] >= 2010) & (df_clean['year'] <= 2017)) | (df_clean['year'] == 2023))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2024))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print("Train: {:,} samples".format(train_mask.sum()))
print("Val:   {:,} samples".format(val_mask.sum()))
print("Test:  {:,} samples".format(test_mask.sum()))

In [ ]:
# Define feature sets
base_features = airline_cols + route_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']
non_weather_features = ['delay_rate_gradient', 'n_public_holidays_total', 'pct_school_holiday']

# Baseline: same as notebook 15 best model (no weather)
features_baseline = base_features + non_weather_features

# With lag12
features_with_lag12 = base_features + ['delay_rate_lag12'] + non_weather_features

print("Baseline features: {}".format(len(features_baseline)))
print("With lag12:        {}".format(len(features_with_lag12)))

## 5. Model Comparison

In [ ]:
def evaluate_model(df, features, train_mask, val_mask, test_mask, name):
    """Train and evaluate Ridge, RF, and XGBoost."""
    X_train = df.loc[train_mask, features].values
    X_val = df.loc[val_mask, features].values
    X_test = df.loc[test_mask, features].values
    
    y_train = df.loc[train_mask, 'delay_rate'].values
    y_val = df.loc[val_mask, 'delay_rate'].values
    y_test = df.loc[test_mask, 'delay_rate'].values
    
    y_train_clf = df.loc[train_mask, 'is_high_delay'].values
    y_val_clf = df.loc[val_mask, 'is_high_delay'].values
    y_test_clf = df.loc[test_mask, 'is_high_delay'].values
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    results = {'name': name}
    
    # Ridge
    ridge = Ridge(alpha=100)
    ridge.fit(X_train_scaled, y_train)
    results['ridge_r2'] = r2_score(y_test, ridge.predict(X_test_scaled))
    
    # Random Forest
    rf = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    results['rf_r2'] = r2_score(y_test, rf.predict(X_test))
    results['rf_model'] = rf
    
    # XGBoost
    if HAS_XGB:
        xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                     min_child_weight=5, random_state=42, n_jobs=-1)
        xgb_clf.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
        xgb_pred = xgb_clf.predict(X_test)
        results['xgb_f1'] = f1_score(y_test_clf, xgb_pred)
    else:
        results['xgb_f1'] = np.nan
    
    return results, features

In [ ]:
# Evaluate both models
print("Training models...")
print()

results_baseline, _ = evaluate_model(df_clean, features_baseline, train_mask, val_mask, test_mask, "Baseline")
results_lag12, features_lag12_used = evaluate_model(df_clean, features_with_lag12, train_mask, val_mask, test_mask, "With Lag12")

print("=" * 70)
print("RESULTS COMPARISON")
print("=" * 70)
print()
header = "{:<15} {:>12} {:>12} {:>12}".format('Model', 'Ridge R2', 'RF R2', 'XGB F1')
print(header)
print("-" * 55)

for r in [results_baseline, results_lag12]:
    xgb_str = "{:.4f}".format(r['xgb_f1']) if not np.isnan(r['xgb_f1']) else 'N/A'
    line = "{:<15} {:>12.4f} {:>12.4f} {:>12}".format(r['name'], r['ridge_r2'], r['rf_r2'], xgb_str)
    print(line)

print()
print("Delta (Lag12 - Baseline):")
print("  Ridge R2: {:+.4f}".format(results_lag12['ridge_r2'] - results_baseline['ridge_r2']))
print("  RF R2:    {:+.4f}".format(results_lag12['rf_r2'] - results_baseline['rf_r2']))
if HAS_XGB:
    print("  XGB F1:   {:+.4f}".format(results_lag12['xgb_f1'] - results_baseline['xgb_f1']))

## 6. Feature Importance Analysis

In [ ]:
# Feature importance from RF with lag12
rf_model = results_lag12['rf_model']
importance_df = pd.DataFrame({
    'Feature': features_with_lag12,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (RF with Lag12):")
print("-" * 50)
for i, row in importance_df.head(15).iterrows():
    marker = " <-- NEW" if row['Feature'] == 'delay_rate_lag12' else ""
    print("  {:<35} {:.4f}{}".format(row['Feature'], row['Importance'], marker))

# Where does lag12 rank?
lag12_rank = importance_df[importance_df['Feature'] == 'delay_rate_lag12'].index[0]
lag12_importance = importance_df[importance_df['Feature'] == 'delay_rate_lag12']['Importance'].values[0]
print()
print("Lag12 rank: {} (importance: {:.4f})".format(
    list(importance_df['Feature']).index('delay_rate_lag12') + 1,
    lag12_importance
))

In [ ]:
# Visualize feature importance
fig, ax = plt.subplots(figsize=(10, 6))

top_features = importance_df.head(12)
y_pos = np.arange(len(top_features))
colors = ['#e74c3c' if f == 'delay_rate_lag12' else 'steelblue' for f in top_features['Feature']]

ax.barh(y_pos, top_features['Importance'], color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_features['Feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Feature Importances (RF with Lag12)')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 7. Conclusions

In [ ]:
print("=" * 70)
print("LAG12 FEATURE EVALUATION")
print("=" * 70)
print()

delta_ridge = results_lag12['ridge_r2'] - results_baseline['ridge_r2']
delta_rf = results_lag12['rf_r2'] - results_baseline['rf_r2']
delta_xgb = results_lag12['xgb_f1'] - results_baseline['xgb_f1'] if HAS_XGB else 0

print("PERFORMANCE IMPACT:")
print("  Ridge R2 delta:  {:+.4f}".format(delta_ridge))
print("  RF R2 delta:     {:+.4f}".format(delta_rf))
print("  XGB F1 delta:    {:+.4f}".format(delta_xgb))
print()

print("DATA COST:")
print("  Rows lost: {:,} ({:.1f}% of baseline)".format(
    len(df_baseline) - len(df_with_lag12),
    (len(df_baseline) - len(df_with_lag12)) / len(df_baseline) * 100
))

### Observations

- Adding 12-month lagged delay (`delay_rate_lag12`) rate is a success, probably the biggest improvement since adding 1-month lagged delay rate (`delay_rate_lag1`)
- Improves all 3 models tested here: particularly the regression models, a marginal improvement for the XGBoost classification model.

### Verdict
`delay_rate_lag12` will be included in the forecasting model.

## 8. Next Step

The next notebook will explore the addition of `load_factor` (explored in notebook 12) to the forecasting model.